In [0]:
df_orders=spark.read.table("ecommerce.bronze.orders")
display(df_orders)

In [0]:
df_orders.printSchema()

In [0]:
from pyspark.sql.functions import *
time_stamp_columns=["order_purchase_timestamp","order_approved_at","order_delivered_carrier_date","order_delivered_customer_date","order_estimated_delivery_date"]
for c in time_stamp_columns:
    df_orders=df_orders.withColumn(c, col(c).cast("TIMESTAMP"))
df_orders.printSchema()

In [0]:
df_orders.select(col("order_status")).distinct().display()

In [0]:
from pyspark.sql.window import Window
window_spec=Window.partitionBy("order_status")
df_orders.groupBy(col("order_status")).agg(count(when(col('order_approved_at').isNull(),1)).alias("approved_nulls"),count(when(col("order_delivered_carrier_date").isNull(),1)).alias("carrier_nulls"),count(when(col("order_delivered_customer_date").isNull(),1)).alias("customer_delivery_nulls")).display()

In [0]:
df_orders=df_orders.withColumn("delivery_date_quality_flag",when((col('order_status')=="delivered") & (col("order_delivered_customer_date").isNull()),"MISSING_EXPECTED_DELIVERY_DATE").otherwise("VALID"))
display(df_orders)

In [0]:
df_orders.filter((col("order_status")=="delivered") & (col("order_approved_at").isNull())).display()

In [0]:
#checking timestamps that can actually follows the chronological order
# order_purchase_timestamp
#         ↓
# order_approved_at
#         ↓
# order_delivered_carrier_date
#         ↓
# order_delivered_customer_date
df_orders.filter(col("order_purchase_timestamp")>col("order_approved_at")).display()
df_orders.filter(col("order_approved_at")>col("order_delivered_carrier_date")).display()
df_orders.filter(col("order_delivered_carrier_date")>col("order_delivered_customer_date")).display()
df_orders.filter(col("order_delivered_customer_date")>col("order_estimated_delivery_date")).display()


In [0]:
# order_approved_at > order_delivered_carrier_date inspecting them
df_orders.filter(col("order_approved_at")>col("order_delivered_carrier_date")).display()

In [0]:
# order_delivered_carrier_date > order_delivered_customer_date inspecting them
df_orders.filter(col("order_delivered_carrier_date")>col("order_delivered_customer_date")).display()

In [0]:
df_orders=df_orders.withColumnRenamed("delivery_date_quality_flag","data_quality_status").withColumn("data_quality_status",when(col("data_quality_status")=="MISSING_EXPECTED_DELIVERY_DATE",lit("MISSING_EXPECTED_DELIVERY_DATE")).when(col("order_purchase_timestamp")>col("order_approved_at"),lit("INVALID_DELIVERY_SEQUENCE")).when(col("order_approved_at")>col("order_delivered_carrier_date"),lit("INVALID_DELIVERY_SEQUENCE")).when(col("order_delivered_carrier_date")>col("order_delivered_customer_date"),lit("INVALID_DELIVERY_SEQUENCE")).when(col("order_delivered_customer_date")>col("order_estimated_delivery_date"),lit("INVALID_DELIVERY_SEQUENCE")).otherwise(lit("VALID")))

In [0]:
df_orders.groupBy("order_status").agg(sum(when(col("order_delivered_customer_date").isNotNull(),1).otherwise(0)).alias("x")).display()

In [0]:
df_orders.select("order_purchase_timestamp","order_approved_at","order_delivered_carrier_date","order_delivered_customer_date","order_estimated_delivery_date","order_status","data_quality_status").filter((col("order_status")=="canceled") & col("order_delivered_customer_date").isNotNull()).display()


In [0]:
df_orders = df_orders.withColumn(
    "data_quality_status",
    concat_ws(
        ",",
        when(
            (col("order_status") == "delivered") &
            col("order_delivered_customer_date").isNull(),
            "MISSING_EXPECTED_DELIVERY_DATE"
        ),
        when(
            (col("order_purchase_timestamp") > col("order_approved_at")) |
            (col("order_approved_at") > col("order_delivered_carrier_date")) |
            (col("order_delivered_carrier_date") > col("order_delivered_customer_date")),
            "INVALID_DELIVERY_SEQUENCE"
        ),
        when(
            (col("order_status") == "canceled") &
            col("order_delivered_customer_date").isNotNull(),
            "CANCELED_WITH_DELIVERY_DATE"
        )
    )
)

df_orders = df_orders.withColumn(
    "data_quality_status",
    when(
        col("data_quality_status") == "",
        "VALID"
    ).otherwise(col("data_quality_status"))
)

In [0]:
display(df_orders)

In [0]:
df_orders.select(
    sum(when(col("data_quality_status").contains("MISSING_EXPECTED_DELIVERY_DATE"), 1).otherwise(0)).alias("MISSING_EXPECTED_DELIVERY_DATE"),
    sum(when(col("data_quality_status").contains("INVALID_DELIVERY_SEQUENCE"), 1).otherwise(0)).alias("INVALID_DELIVERY_SEQUENCE"),
    sum(when(col("data_quality_status").contains("CANCELED_WITH_DELIVERY_DATE"), 1).otherwise(0)).alias("CANCELED_WITH_DELIVERY_DATE")
).display()

In [0]:
df_orders.filter(col("data_quality_status").contains("CANCELED_WITH_DELIVERY_DATE")&col("customer_id").contains("INVALID_DELIVERY_SEQUENCE")).display()

In [0]:
df_orders.groupBy("data_quality_status").count().display()

In [0]:
total_rows=df_orders.count()
distinct_rows=df_orders.select("order_id").distinct().count()
print("Total rows: ",total_rows)
print("Distinct rows: ",distinct_rows)
print("Percentage of distinct rows: ",distinct_rows/total_rows*100)

In [0]:
df_orders.printSchema()


In [0]:
df_orders.groupBy("data_quality_status").count().display()

In [0]:
df_orders.write.format("delta").mode("overwrite").saveAsTable("ecommerce.silver.orders")

### customers

In [0]:
df_customers=spark.read.table("ecommerce.bronze.customers")

In [0]:
display(df_customers)

In [0]:
df_customers.select("customer_state","customer_city","customer_zip_code_prefix").display()


In [0]:
from pyspark.sql.functions import *

df_customers.select("customer_city").distinct().sort("customer_city").display()

In [0]:
df_customers.filter(
    lower(col("customer_city")).contains("sao paulo")
).select("customer_city").distinct().display()

In [0]:
df_customers.select("customer_state").distinct().display()

In [0]:
df_customers.filter(col("customer_state")!=trim("customer_state")).count()

In [0]:
df_customers.select("customer_zip_code_prefix").distinct().display()

In [0]:
df_customers.filter(col("customer_zip_code_prefix")!=trim(col("customer_zip_code_prefix"))).count()

In [0]:
df_customers.write.format("delta").mode("overwrite").saveAsTable("ecommerce.silver.customers")

### sellers

In [0]:
df_sellers=spark.read.table("ecommerce.bronze.sellers")
display(df_sellers)

In [0]:
df_sellers.filter(col("seller_state")!=trim(col("seller_state"))).count()

In [0]:
df_sellers.filter(col("seller_state")!=upper("seller_state")).count()

In [0]:
df_sellers.select("seller_state").distinct().display()

In [0]:
df_sellers.select("seller_zip_code_prefix").distinct().display()

In [0]:
df_sellers.write.format("delta").mode("overwrite").saveAsTable("ecommerce.silver.sellers")

### order_items

In [0]:
df_order_items=spark.read.table("ecommerce.bronze.order_items")
display(df_order_items)

In [0]:
df_order_items=df_order_items.withColumn("shipping_limit_date",col("shipping_limit_date").cast("timestamp"))
df_order_items.printSchema()

In [0]:
df_order_items.filter(col("price").cast("double")>0).count()

In [0]:


df_order_items.select(
    "price",
    (
        length(col("price")) - instr(col("price"), ".")
    ).alias("decimal_places")
).groupBy("decimal_places").count().orderBy("decimal_places").display()

In [0]:
df_order_items=df_order_items.withColumn("price",col("price").cast("double"))

In [0]:
df_order_items.select("freight_value").display()

In [0]:
df_order_items.select("freight_value",(length("freight_value")-instr("freight_value",".")).alias("decimal")).groupBy("decimal").agg(count("decimal")).display()

In [0]:
df_order_items=df_order_items.withColumn("freight_value",col("freight_value").cast("double"))
df_order_items.printSchema()

In [0]:
df_order_items.filter(col("freight_value")<0).count()

In [0]:
df_order_items.write.format("delta").mode("overwrite").saveAsTable("ecommerce.silver.order_items")

### products

In [0]:
df_products=spark.read.table("ecommerce.bronze.products")
display(df_products)

In [0]:
from pyspark.sql.functions import *
df_products.filter(col("product_category_name").isNull()).select("product_name_lenght","product_description_lenght","product_photos_qty").display()

In [0]:
df_products.filter(col("product_weight_g").isNull()).select("product_weight_g",
"product_length_cm",
"product_height_cm",
"product_width_cm").display()

In [0]:

df_products.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in [
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
]).display()

In [0]:
columns=[
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
for c in columns:
    df_products=df_products.withColumn(c,col(c).cast("int"))
df_products.printSchema()

In [0]:
columns=[
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
x=[]
for c in columns:
    x.append([c,df_products.filter(col(c)<0).count()])
print(x)

In [0]:
df_category_name_translation=spark.read.table("ecommerce.bronze.category_name_translation")
display(df_category_name_translation)


In [0]:
df_products.join(
    df_category_name_translation,
    df_products.product_category_name == df_category_name_translation.product_category_name,
    "left"
).filter(
    df_products.product_category_name.isNotNull() &
    df_category_name_translation.product_category_name_english.isNull()
).select(
    df_products.product_category_name,
    df_category_name_translation.product_category_name,
    df_category_name_translation.product_category_name_english
).distinct().display()

In [0]:

x=df_category_name_translation.count()
y=df_category_name_translation.select(col("product_category_name")).distinct().count()
print(x-y)

In [0]:
df_products.write.format("delta").mode("overwrite").saveAsTable("ecommerce.silver.products")

In [0]:
df_category=spark.read.table("ecommerce.bronze.category_name_translation")
display(df_category)

In [0]:
from pyspark.sql.functions import *
x=[]
for c in df_category.columns:
    x.append([c,df_category.filter(col(c)!=trim(c)).count()])
print(x)

In [0]:
df_category.filter(
    (col("product_category_name") != lower(col("product_category_name"))) |
    (col("product_category_name_english") != lower(col("product_category_name_english")))
).display()

In [0]:
df_category.write.format("delta").mode("overwrite").saveAsTable("ecommerce.silver.category_name_translation")

In [0]:
df_order_payments=spark.read.table("ecommerce.bronze.order_payments")
display(df_order_payments)

In [0]:
df_order_payments.filter(col("payment_value")!=col("payment_value").cast("double")).count()

In [0]:
df_order_payments=df_order_payments.withColumn("payment_sequential",col("payment_sequential").cast("int")).withColumn("payment_installments",col("payment_installments").cast("int")).withColumn("payment_value",col("payment_value").cast("double"))

In [0]:
df_order_payments.printSchema()

In [0]:
df_order_payments.filter(
    (col("payment_sequential") < 0) |
    (col("payment_installments") < 0) |
    (col("payment_value") < 0)
).display()

In [0]:
df_order_payments.filter(col("payment_value")!=round(col("payment_value"),2)).display()

In [0]:
df_order_payments.select("payment_type").distinct().display()

In [0]:
df_order_payments.select("order_id","payment_sequential").distinct().count()

In [0]:
df_order_payments.write.format("delta").mode("overwrite").saveAsTable("ecommerce.silver.order_payments")